# P4 -- Habitat Suitability Exploration

This notebook demonstrates the habitat suitability modelling workflow
for Pacific fisher (*Pekania pennanti*) in the Sierra Nevada:

1. Load and map species occurrence records
2. Visualise environmental predictor rasters
3. Train and evaluate SDMs (MaxEnt, Random Forest)
4. Display current and future suitability surfaces
5. Analyse habitat change (gain / loss / refugia)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Ensure project root is on the path
PROJECT_ROOT = Path.cwd().resolve().parents[2]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared.data.generate_synthetic import (
    generate_synthetic_occurrences,
    generate_synthetic_predictors,
)

## 1. Generate synthetic data and load occurrences

In [ ]:
DATA_DIR = Path("../data/scratch")
DATA_DIR.mkdir(parents=True, exist_ok=True)

occ_path = generate_synthetic_occurrences(DATA_DIR)
pred_paths = generate_synthetic_predictors(DATA_DIR)

config = {
    "species": {"name": "Pekania pennanti", "thinning_distance_km": 0.01},
    "data": {
        "occurrences_path": str(occ_path),
        "predictor_dir": str(DATA_DIR),
        "output_dir": str(DATA_DIR / "output"),
    },
    "modeling": {
        "crs": "EPSG:3310",
        "algorithms": ["maxent", "random_forest"],
        "cv_folds": 5,
    },
}

from projects.p4_habitat_suitability.src.occurrences import load_occurrences

occ = load_occurrences(config)
print(f"Loaded {len(occ)} occurrence records")
occ.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
occ.plot(ax=ax, column="source", legend=True, markersize=20, cmap="Set1")
ax.set_title("Pacific Fisher Occurrence Records")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.tight_layout()
plt.show()

## 2. Visualise predictor rasters

In [ ]:
from projects.p4_habitat_suitability.src.predictors import build_predictor_stack

stack, profile, band_names = build_predictor_stack(config)
print(f"Stack shape: {stack.shape}")
print(f"Band names: {band_names}")

transform = profile["transform"]
extent = [
    transform.c,
    transform.c + stack.shape[2] * transform.a,
    transform.f + stack.shape[1] * transform.e,
    transform.f,
]

n_bands = len(band_names)
ncols = 3
nrows = int(np.ceil(n_bands / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 5 * nrows))
axes = axes.flatten()

for i, name in enumerate(band_names):
    im = axes[i].imshow(stack[i], cmap="viridis", extent=extent, origin="upper")
    axes[i].set_title(name)
    plt.colorbar(im, ax=axes[i], shrink=0.7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## 3. Train models and evaluate performance

In [ ]:
from projects.p4_habitat_suitability.src.background import (
    create_pa_matrix,
    generate_background_points,
)
from projects.p4_habitat_suitability.src.modeling import (
    compute_variable_importance,
    spatial_block_cv,
    train_maxent,
    train_random_forest,
)
from projects.p4_habitat_suitability.src.predictors import extract_values_at_points

bg = generate_background_points(occ, stack, profile, config, n_points=500)
X, y = create_pa_matrix(occ, bg, stack, profile, band_names)
print(f"PA matrix: {X.shape[0]} samples, {X.shape[1]} predictors")

# Train models
maxent_model = train_maxent(X, y, config)
rf_model = train_random_forest(X, y, config)
print("Models trained successfully.")

In [ ]:
# Variable importance (Random Forest)
vimp = compute_variable_importance(rf_model, X, y, band_names)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(vimp["variable"], vimp["importance"], color="steelblue")
ax.set_xlabel("Permutation Importance (AUC drop)")
ax.set_title("Variable Importance -- Random Forest")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Spatial block cross-validation
pres_df = extract_values_at_points(stack, profile, occ, band_names)
bg_df = extract_values_at_points(stack, profile, bg, band_names)
coords = np.vstack([pres_df[["x", "y"]].values, bg_df[["x", "y"]].values])[:len(X)]

cv_rf = spatial_block_cv(
    X, y, coords,
    model_fn=lambda Xt, yt: train_random_forest(Xt, yt),
    config=config,
)
print(f"RF AUC: {cv_rf['auc_mean']:.3f} +/- {cv_rf['auc_std']:.3f}")
print(f"RF TSS: {cv_rf['tss_mean']:.3f} +/- {cv_rf['tss_std']:.3f}")

## 4. Current and future suitability maps

In [ ]:
from projects.p4_habitat_suitability.src.projection import (
    project_suitability,
)

suit_current, suit_profile = project_suitability(rf_model, stack, profile)

# Simulate a modest future shift by perturbing temperature band
future_stack = stack.copy()
temp_idx = band_names.index("bio1_mean_temp")
future_stack[temp_idx] += 2.0  # +2 C warming

suit_future, _ = project_suitability(rf_model, future_stack, profile)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, data, title in zip(
    axes,
    [suit_current, suit_future],
    ["Current Suitability", "Future Suitability (SSP2-4.5, 2050)"],
):
    im = ax.imshow(data, cmap="RdYlGn", vmin=0, vmax=1, extent=extent, origin="upper")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label="Probability")
plt.tight_layout()
plt.show()

## 5. Change analysis -- gain / loss / refugia

In [ ]:
from matplotlib.colors import ListedColormap

from projects.p4_habitat_suitability.src.change_analysis import (
    compute_change,
    summarize_change,
)

change = compute_change(suit_current, suit_future)
change_df = summarize_change(change, suit_profile)
display(change_df)

cmap = ListedColormap(["#d9d9d9", "#2ca02c", "#1f77b4", "#d62728"])
labels = ["Stable Unsuitable", "Refugia", "Gain", "Loss"]

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(change, cmap=cmap, vmin=0, vmax=3, extent=extent, origin="upper")
cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2, 3])
cbar.ax.set_yticklabels(labels)
ax.set_title("Habitat Change: Current vs. Future")
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated the end-to-end habitat suitability workflow on
synthetic data for Pacific fisher.  The same pipeline functions are
available as importable modules and can be driven by the YAML
configuration via `pipeline.run_pipeline()`.